In [4]:
import random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# [재현성 시드 고정]
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# ===============================
# [STEP 1] 고도화된 Feature Engineering
# ===============================
def advanced_fe(df):
    df = df.copy()
    
    # 1. 마키아벨리즘 역채점 및 점수 (심리학 도메인)
    rev_cols = ['QaA', 'QdA', 'QgA', 'QiA', 'QlA', 'QnA', 'QpA', 'QrA', 'QtA']
    for col in rev_cols:
        df[col] = 6 - df[col]
    
    q_cols = [c for c in df.columns if c.endswith('A')]
    df['mach_score'] = df[q_cols].mean(axis=1)
    
    # 2. Big5 성격 요인 추가
    df['tp_ext'] = df['tp01'] - df['tp06']
    df['tp_est'] = df['tp09'] - df['tp04']
    df['tp_agr'] = df['tp07'] - df['tp02']
    df['tp_csn'] = df['tp03'] - df['tp08']
    df['tp_opn'] = df['tp05'] - df['tp10']
    
    # 3. 시간 데이터 처리 (로그 변환)
    time_cols = [c for c in df.columns if c.endswith('E')]
    for col in time_cols:
        df[col] = np.log1p(df[col].clip(lower=0, upper=1e6))
    
    # 4. 불필요한 칼럼 제거
    drop_cols = ['index', 'hand']
    df = df.drop(drop_cols, axis=1)
    
    # 5. 카테고리 변수 처리
    cat_cols = ['education', 'engnat', 'married', 'urban', 'gender', 'race', 'religion', 'age_group']
    df = pd.get_dummies(df, columns=cat_cols)
    
    return df

train_raw = pd.read_csv('train.csv')
test_raw = pd.read_csv('test_x.csv')

# 이상치 제거 및 전처리
train_raw = train_raw.drop(train_raw[train_raw.familysize > 50].index)
train_y = (2 - train_raw['voted'].to_numpy()).astype(np.float32)

train_x = advanced_fe(train_raw.drop('voted', axis=1))
test_x = advanced_fe(test_raw)

# 스케일링 (신경망 수렴을 위해 필수)
scaler = StandardScaler()
train_x = scaler.fit_transform(train_x).astype(np.float32)
test_x = scaler.transform(test_x).astype(np.float32)

# 텐서 변환
train_x_t = torch.tensor(train_x)
train_y_t = torch.tensor(train_y)
test_x_t = torch.tensor(test_x)

# ===============================
# [STEP 3] Residual MLP 모델 정의
# ===============================
class ResBlock(nn.Module):
    def __init__(self, size, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(size, size),
            nn.BatchNorm1d(size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(size, size),
            nn.BatchNorm1d(size)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x)) # 잔차 연결

class AdvancedDeepMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU()
        )
        self.res_layers = nn.Sequential(
            ResBlock(512, 0.3),
            ResBlock(512, 0.3)
        )
        self.output_layer = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.input_layer(x)
        x = self.res_layers(x)
        return self.output_layer(x)

In [5]:
# 가중치 앙상블
sub_autogluon = pd.read_csv('final_masterpiece_ensemble_2.csv')['voted']
sub_advanced_mlp = pd.read_csv('submission_1st_winner_inverted.csv')['voted']

# 성능이 더 좋은 쪽에 가중치를 더 줌
final_voted = (sub_autogluon * 0.4) + (sub_advanced_mlp * 0.6)

submission = pd.read_csv('sample_submission.csv')
submission['voted'] = final_voted
submission.to_csv('final_masterpiece_ensemble_3.csv', index=False)                                